## 1. Setup & Imports

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import time
import warnings
warnings.filterwarnings('ignore')

# Import custom LSTM predictor
from LSTM_Price_Predictor import LSTMPricePredictor
from Data_Handling import get_data_auto

print("Libraries imported successfully!")
print(f"Current date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

Libraries imported successfully!
Current date: 2026-01-20 14:29:40


## 2. Data Configuration

In [4]:
# Symbol and timeframe
SYMBOL = "SPY"
TIMEFRAME_UNIT = "Minute"
MULTIPLIER = 1  # 1-minute bars

# Training period - 3 months for robust parameter selection
TRAIN_START = "2024-10-01"
TRAIN_END = "2024-12-31"  # 3 months (~23,400 minute bars)

# Validation period - 1 month
VAL_START = "2025-01-01"
VAL_END = "2025-01-31"  # 1 month (~7,800 minute bars)

print(f"Data Configuration:")
print(f"  Symbol: {SYMBOL}")
print(f"  Training: {TRAIN_START} to {TRAIN_END} (3 months)")
print(f"  Validation: {VAL_START} to {VAL_END} (1 month)")
print(f"  Timeframe: {MULTIPLIER}-minute bars")

Data Configuration:
  Symbol: SPY
  Training: 2024-10-01 to 2024-12-31 (3 months)
  Validation: 2025-01-01 to 2025-01-31 (1 month)
  Timeframe: 1-minute bars


## 3. Grid Search Parameter Space

**Medium Search (~12-15 combinations)**:
- Focus on most impactful parameters
- Balance between thoroughness and computation time
- Estimated runtime: 2-3 hours

In [5]:
# Define parameter grid
param_grid = {
    # Architecture parameters
    'lstm_units': [50, 64, 100],           # Neurons per layer
    'num_layers': [1, 2],                  # LSTM depth
    'dropout': [0.2, 0.3],                 # Regularization
    
    # Temporal parameters
    'lookback_window': [390, 585, 780],    # 1, 1.5, 2 trading days
    
    # Training parameters (fixed for consistency)
    'epochs': [30],                        # Proper training with early stopping
    'batch_size': [64],                    # Standard batch size
    'learning_rate': [0.001],              # Adam default
    
    # Prediction (always 1 for grid search)
    'prediction_horizon': [1]              # Single-step only
}

# Generate all combinations (will filter to ~12-15 best candidates)
from itertools import product

# Create strategic subset instead of full grid
search_configs = [
    # Small models (fast, baseline)
    {'lstm_units': 50, 'num_layers': 1, 'dropout': 0.2, 'lookback_window': 390},
    {'lstm_units': 50, 'num_layers': 1, 'dropout': 0.2, 'lookback_window': 585},
    
    # Medium models (balanced)
    {'lstm_units': 64, 'num_layers': 1, 'dropout': 0.2, 'lookback_window': 585},
    {'lstm_units': 64, 'num_layers': 2, 'dropout': 0.2, 'lookback_window': 585},
    {'lstm_units': 64, 'num_layers': 2, 'dropout': 0.3, 'lookback_window': 585},
    
    # Larger models (capacity)
    {'lstm_units': 100, 'num_layers': 1, 'dropout': 0.2, 'lookback_window': 585},
    {'lstm_units': 100, 'num_layers': 2, 'dropout': 0.2, 'lookback_window': 585},
    {'lstm_units': 100, 'num_layers': 2, 'dropout': 0.3, 'lookback_window': 585},
    
    # Longer lookback tests
    {'lstm_units': 64, 'num_layers': 2, 'dropout': 0.2, 'lookback_window': 780},
    {'lstm_units': 100, 'num_layers': 2, 'dropout': 0.2, 'lookback_window': 780},
    
    # Shorter lookback tests
    {'lstm_units': 64, 'num_layers': 2, 'dropout': 0.2, 'lookback_window': 390},
    {'lstm_units': 100, 'num_layers': 2, 'dropout': 0.3, 'lookback_window': 390},
]

# Add fixed parameters to each config
for config in search_configs:
    config.update({
        'epochs': 30,
        'batch_size': 64,
        'learning_rate': 0.001,
        'prediction_horizon': 1
    })

print(f"Grid Search Configuration:")
print(f"  Total configurations to test: {len(search_configs)}")
print(f"  Estimated time per config: 10-15 minutes")
print(f"  Total estimated runtime: {len(search_configs) * 12 / 60:.1f} hours")
print(f"\nParameter ranges:")
print(f"  LSTM Units: {sorted(set(c['lstm_units'] for c in search_configs))}")
print(f"  Num Layers: {sorted(set(c['num_layers'] for c in search_configs))}")
print(f"  Dropout: {sorted(set(c['dropout'] for c in search_configs))}")
print(f"  Lookback: {sorted(set(c['lookback_window'] for c in search_configs))}")

Grid Search Configuration:
  Total configurations to test: 12
  Estimated time per config: 10-15 minutes
  Total estimated runtime: 2.4 hours

Parameter ranges:
  LSTM Units: [50, 64, 100]
  Num Layers: [1, 2]
  Dropout: [0.2, 0.3]
  Lookback: [390, 585, 780]


## 4. Load Training & Validation Data

In [6]:
print("Loading training data (Oct-Dec 2024)...")
train_data = get_data_auto(
    symbol=SYMBOL,
    start=TRAIN_START,
    end=TRAIN_END,
    timeframe_unit=TIMEFRAME_UNIT,
    multiplier=MULTIPLIER
)

print(f"\nLoading validation data (Jan 2025)...")
val_data = get_data_auto(
    symbol=SYMBOL,
    start=VAL_START,
    end=VAL_END,
    timeframe_unit=TIMEFRAME_UNIT,
    multiplier=MULTIPLIER
)

print(f"\nData loaded successfully:")
print(f"  Training samples: {len(train_data):,} bars")
print(f"  Training period: {train_data.index[0]} to {train_data.index[-1]}")
print(f"  Validation samples: {len(val_data):,} bars")
print(f"  Validation period: {val_data.index[0]} to {val_data.index[-1]}")
print(f"\nFeatures: {list(train_data.columns)}")

Loading training data (Oct-Dec 2024)...

Loading validation data (Jan 2025)...

Loading validation data (Jan 2025)...

Data loaded successfully:
  Training samples: 48,931 bars
  Training period: 2024-10-01 08:00:00+00:00 to 2024-12-31 00:00:00+00:00
  Validation samples: 15,254 bars
  Validation period: 2025-01-01 00:00:00+00:00 to 2025-01-30 23:59:00+00:00

Features: ['Open', 'High', 'Low', 'Close', 'Volume']

Data loaded successfully:
  Training samples: 48,931 bars
  Training period: 2024-10-01 08:00:00+00:00 to 2024-12-31 00:00:00+00:00
  Validation samples: 15,254 bars
  Validation period: 2025-01-01 00:00:00+00:00 to 2025-01-30 23:59:00+00:00

Features: ['Open', 'High', 'Low', 'Close', 'Volume']


## 5. Grid Search Execution

Train each configuration and collect metrics:
- Training loss & MAE
- Validation loss & MAE
- Directional accuracy (most important!)
- Training time
- Model complexity (parameters)

In [8]:
# Storage for results
results = []

print("="*80)
print("STARTING GRID SEARCH")
print("="*80)

for idx, config in enumerate(search_configs, 1):
    print(f"\n{'='*80}")
    print(f"Configuration {idx}/{len(search_configs)}")
    print(f"{'='*80}")
    print(f"  LSTM Units: {config['lstm_units']}")
    print(f"  Num Layers: {config['num_layers']}")
    print(f"  Dropout: {config['dropout']}")
    print(f"  Lookback: {config['lookback_window']} minutes ({config['lookback_window']/390:.1f} days)")
    
    start_time = time.time()
    
    try:
        # Initialize model with current configuration
        model = LSTMPricePredictor(
            lookback_window=config['lookback_window'],
            prediction_horizon=config['prediction_horizon'],
            lstm_units=config['lstm_units'],
            num_layers=config['num_layers'],
            dropout=config['dropout'],
            epochs=config['epochs'],
            batch_size=config['batch_size'],
            learning_rate=config['learning_rate'],
            prediction_mode='recursive',
            verbose=0  # Suppress training output for cleaner logs
        )
        
        # Train model
        print(f"\n  Training...")
        history = model.fit(
            train_data,
            validation_split=0.2,
            early_stopping_patience=10
        )
        
        # Generate predictions on validation data
        print(f"  Generating predictions...")
        predictions = model.predict(val_data)
        
        # Evaluate performance
        eval_results = model.evaluate(predictions=predictions)
        dir_accuracy = model.get_directional_accuracy(predictions=predictions)
        
        # Calculate training time
        train_time = time.time() - start_time
        
        # Extract key metrics
        result = {
            'config_id': idx,
            'lstm_units': config['lstm_units'],
            'num_layers': config['num_layers'],
            'dropout': config['dropout'],
            'lookback_window': config['lookback_window'],
            'train_loss': eval_results.loc['pred_price_1', 'RMSE'],  # Use RMSE as proxy for loss
            'val_loss': eval_results.loc['pred_price_1', 'RMSE'],
            'val_mae': eval_results.loc['pred_price_1', 'MAE'],
            'val_mape': eval_results.loc['pred_price_1', 'MAPE'],
            'directional_accuracy': dir_accuracy.loc['pred_price_1', 'Accuracy (%)'],
            'train_time_minutes': train_time / 60,
            'total_params': config['lstm_units'] * config['num_layers']  # Approximate
        }
        
        results.append(result)
        
        # Print summary
        print(f"\n  Results:")
        print(f"    Val Loss (RMSE): {result['val_loss']:.4f}")
        print(f"    Val MAE: {result['val_mae']:.4f}")
        print(f"    Directional Accuracy: {result['directional_accuracy']:.2f}%")
        print(f"    Training Time: {result['train_time_minutes']:.1f} minutes")
        print(f"  ✓ Success")
        
    except Exception as e:
        print(f"  ✗ Failed: {str(e)}")
        # Log failure but continue
        results.append({
            'config_id': idx,
            'lstm_units': config['lstm_units'],
            'num_layers': config['num_layers'],
            'dropout': config['dropout'],
            'lookback_window': config['lookback_window'],
            'error': str(e)
        })

print(f"\n{'='*80}")
print("GRID SEARCH COMPLETE")
print(f"{'='*80}")

STARTING GRID SEARCH

Configuration 1/12
  LSTM Units: 50
  Num Layers: 1
  Dropout: 0.2
  Lookback: 390 minutes (1.0 days)

  Training...
  Generating predictions...
  Generating predictions...
  ✗ Failed: 'pred_price_1'

Configuration 2/12
  LSTM Units: 50
  Num Layers: 1
  Dropout: 0.2
  Lookback: 585 minutes (1.5 days)

  Training...
  ✗ Failed: 'pred_price_1'

Configuration 2/12
  LSTM Units: 50
  Num Layers: 1
  Dropout: 0.2
  Lookback: 585 minutes (1.5 days)

  Training...
  Generating predictions...
  Generating predictions...
  ✗ Failed: 'pred_price_1'

Configuration 3/12
  LSTM Units: 64
  Num Layers: 1
  Dropout: 0.2
  Lookback: 585 minutes (1.5 days)

  Training...
  ✗ Failed: 'pred_price_1'

Configuration 3/12
  LSTM Units: 64
  Num Layers: 1
  Dropout: 0.2
  Lookback: 585 minutes (1.5 days)

  Training...


KeyboardInterrupt: 

## 6. Results Analysis & Ranking

In [ ]:
# Convert results to DataFrame
results_df = pd.DataFrame(results)

# Filter out failed runs
successful_results = results_df[~results_df['val_loss'].isna()].copy()

# Rank by directional accuracy (most important metric)
successful_results = successful_results.sort_values('directional_accuracy', ascending=False)
successful_results['rank'] = range(1, len(successful_results) + 1)

print("="*80)
print("GRID SEARCH RESULTS - RANKED BY DIRECTIONAL ACCURACY")
print("="*80)
print(successful_results[[
    'rank', 'lstm_units', 'num_layers', 'dropout', 'lookback_window',
    'directional_accuracy', 'val_loss', 'val_mae', 'train_time_minutes'
]].to_string(index=False))

# Identify best configuration
best_config = successful_results.iloc[0]
print(f"\n{'='*80}")
print("BEST CONFIGURATION (Highest Directional Accuracy)")
print(f"{'='*80}")
print(f"  LSTM Units: {int(best_config['lstm_units'])}")
print(f"  Num Layers: {int(best_config['num_layers'])}")
print(f"  Dropout: {best_config['dropout']}")
print(f"  Lookback Window: {int(best_config['lookback_window'])} minutes ({best_config['lookback_window']/390:.1f} days)")
print(f"\n  Performance:")
print(f"    Directional Accuracy: {best_config['directional_accuracy']:.2f}%")
print(f"    Validation Loss: {best_config['val_loss']:.4f}")
print(f"    Validation MAE: {best_config['val_mae']:.4f}")
print(f"    Training Time: {best_config['train_time_minutes']:.1f} minutes")

## 7. Visualizations

In [ ]:
# Create comprehensive visualization
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. Directional Accuracy by Configuration
ax1 = axes[0, 0]
ax1.barh(range(len(successful_results)), successful_results['directional_accuracy'], color='steelblue')
ax1.axvline(x=50, color='red', linestyle='--', linewidth=2, label='Random (50%)')
ax1.set_xlabel('Directional Accuracy (%)', fontweight='bold')
ax1.set_ylabel('Configuration', fontweight='bold')
ax1.set_title('Directional Accuracy by Configuration', fontweight='bold', fontsize=14)
ax1.set_yticks(range(len(successful_results)))
ax1.set_yticklabels([f"Config {i}" for i in successful_results['config_id']])
ax1.legend()
ax1.grid(axis='x', alpha=0.3)

# 2. Validation Loss vs Accuracy
ax2 = axes[0, 1]
scatter = ax2.scatter(successful_results['val_loss'], successful_results['directional_accuracy'],
                      s=successful_results['lstm_units']*2, alpha=0.6, c=successful_results['num_layers'],
                      cmap='viridis')
ax2.set_xlabel('Validation Loss (RMSE)', fontweight='bold')
ax2.set_ylabel('Directional Accuracy (%)', fontweight='bold')
ax2.set_title('Loss vs Accuracy (size=units, color=layers)', fontweight='bold', fontsize=14)
ax2.axhline(y=50, color='red', linestyle='--', linewidth=1, alpha=0.5)
plt.colorbar(scatter, ax=ax2, label='Num Layers')
ax2.grid(alpha=0.3)

# 3. Training Time vs Performance
ax3 = axes[1, 0]
ax3.scatter(successful_results['train_time_minutes'], successful_results['directional_accuracy'],
            s=100, alpha=0.6, c='coral')
ax3.set_xlabel('Training Time (minutes)', fontweight='bold')
ax3.set_ylabel('Directional Accuracy (%)', fontweight='bold')
ax3.set_title('Training Time vs Accuracy', fontweight='bold', fontsize=14)
ax3.axhline(y=50, color='red', linestyle='--', linewidth=1, alpha=0.5)
ax3.grid(alpha=0.3)

# 4. Lookback Window Impact
ax4 = axes[1, 1]
lookback_grouped = successful_results.groupby('lookback_window')['directional_accuracy'].mean()
ax4.bar(lookback_grouped.index, lookback_grouped.values, color='mediumseagreen', alpha=0.7)
ax4.axhline(y=50, color='red', linestyle='--', linewidth=2, label='Random (50%)')
ax4.set_xlabel('Lookback Window (minutes)', fontweight='bold')
ax4.set_ylabel('Avg Directional Accuracy (%)', fontweight='bold')
ax4.set_title('Impact of Lookback Window', fontweight='bold', fontsize=14)
ax4.set_xticks(lookback_grouped.index)
ax4.set_xticklabels([f"{int(x)}\n({x/390:.1f}d)" for x in lookback_grouped.index])
ax4.legend()
ax4.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## 8. Save Results for Monthly Validation

In [ ]:
# Save full results
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
results_file = f'grid_search_results_{timestamp}.csv'
successful_results.to_csv(results_file, index=False)
print(f"✓ Full results saved to: {results_file}")

# Save best configuration separately for easy reference
best_config_dict = {
    'lstm_units': int(best_config['lstm_units']),
    'num_layers': int(best_config['num_layers']),
    'dropout': float(best_config['dropout']),
    'lookback_window': int(best_config['lookback_window']),
    'epochs': 30,
    'batch_size': 64,
    'learning_rate': 0.001,
    'prediction_horizon': 1,  # Train on 1, test recursively
    'directional_accuracy': float(best_config['directional_accuracy']),
    'val_loss': float(best_config['val_loss'])
}

import json
best_config_file = 'best_architecture_config.json'
with open(best_config_file, 'w') as f:
    json.dump(best_config_dict, f, indent=2)
print(f"✓ Best configuration saved to: {best_config_file}")

print(f"\n{'='*80}")
print("NEXT STEPS:")
print(f"{'='*80}")
print(f"1. Review the best configuration above")
print(f"2. Use these parameters in '02_Monthly_Validation.ipynb'")
print(f"3. Test recursive multi-step predictions (1, 5, 10 minutes)")
print(f"4. Run monthly to validate on new data")
print(f"5. Re-run grid search quarterly to adapt to market changes")